# Prétraitement des données scRNA-seq (pancréas), sans fuite d'information

Ordre des étapes :

| # | Étape | Estimée sur |
|---|---|---|
| 1 | Chargement, comptages bruts | - |
| 2 | Nettoyage des résultats pré-calculés sur toutes les cellules | - |
| 3 | Filtre des cellules (min 200 gènes) | chaque cellule seule |
| 4 | **Séparation train / validation** | - |
| 5 | Filtre des gènes (min 3 cellules) | train |
| 6 | Normalisation (CP10k) puis `log1p` | chaque cellule seule |
| 7 | Sélection des gènes variables (HVG) | train |
| 8 | Sauvegarde de `train.h5ad` et `val.h5ad` | - |
| 9 | Visualisation (PCA du train) | train |

Règle : tout ce qui **apprend** quelque chose sur les données est ajusté sur le train uniquement, puis appliqué à la validation.

## 1. Imports et paramètres

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import scanpy as sc
from sklearn.model_selection import train_test_split

SEED = 0             # graine : même découpage à chaque exécution
N_HVG = 2000         # nombre de gènes variables conservés (à justifier et à tester)
TEST_SIZE = 0.2      # part de cellules mise en validation
DATA_DIR = "data"    # "../data" si le notebook est dans un sous-dossier

np.random.seed(SEED)
os.makedirs(DATA_DIR, exist_ok=True)

## 2. Chargement des données

Le fichier `dataset.h5ad` doit être dans le dossier `data/`. Pour le télécharger (bucket public) :

```bash
aws s3 cp --no-sign-request \
  s3://openproblems-data/resources/datasets/openproblems_v1/pancreas/log_cp10k/dataset.h5ad \
  data/dataset.h5ad
```

In [ ]:
chemin = f"{DATA_DIR}/dataset.h5ad"
assert os.path.exists(chemin), f"Fichier introuvable : {chemin}"

adata = sc.read_h5ad(chemin)
print(adata)

## 3. Comptages bruts dans `adata.X`

Dans ce fichier `adata.X` est vide (`None`) : les données sont dans les layers. On repart des **comptages bruts** (`counts`) pour faire nous-mêmes la normalisation, après le découpage.

In [ ]:
print("X avant :", type(adata.X))
adata.X = adata.layers["counts"].copy()
print("X après :", type(adata.X), adata.X.shape)

## 4. Suppression des résultats pré-calculés

Le fichier contient déjà une PCA, un graphe de voisins, des gènes variables (`hvg`) et des layers normalisées. Tout cela a été calculé sur **toutes** les cellules : le garder reviendrait à laisser la validation influencer le train. On le supprime et on recalcule ce dont on a besoin.

In [ ]:
adata.obsm.clear()
adata.varm.clear()
adata.obsp.clear()
for k in ["knn", "pca_variance"]:
    adata.uns.pop(k, None)

colonnes = [c for c in ["hvg", "hvg_score", "n_cells"] if c in adata.var.columns]
adata.var.drop(columns=colonnes, inplace=True)

for nom in ["normalized", "log_normalized"]:
    if nom in adata.layers:
        del adata.layers[nom]

print(adata)

## 5. Contrôle qualité

On regarde le nombre de gènes détectés et le total de comptages par cellule, pour justifier le seuil du filtre.

In [ ]:
sc.pp.calculate_qc_metrics(adata, inplace=True)
print(adata.obs[["n_genes_by_counts", "total_counts"]].describe())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(adata.obs["n_genes_by_counts"], bins=60)
ax[0].set_xlabel("gènes détectés par cellule")
ax[1].hist(np.log10(adata.obs["total_counts"]), bins=60)
ax[1].set_xlabel("log10(total de comptages)")
plt.tight_layout()
plt.show()

## 6. Filtre des cellules

On retire les cellules avec moins de 200 gènes détectés (cellules de mauvaise qualité). Ce critère ne dépend que de chaque cellule : on peut le faire avant le découpage sans fuite.

In [ ]:
n0 = adata.n_obs
sc.pp.filter_cells(adata, min_genes=200)
print(f"{n0 - adata.n_obs} cellules retirées, {adata.n_obs} restantes")

## 7. Séparation train / validation

On sépare les **indices des cellules** avec `train_test_split`, en gardant les proportions de types cellulaires (`stratify`), puis on crée deux objets AnnData distincts. C'est l'étape clé : tout ce qui suit estime ses paramètres sur le train.

In [ ]:
idx = np.arange(adata.n_obs)

effectifs = adata.obs["celltype"].value_counts()
stratif = adata.obs["celltype"] if effectifs.min() >= 2 else None
if stratif is None:
    print("Un type cellulaire n'a qu'une cellule : pas de stratification")

idx_train, idx_val = train_test_split(
    idx, test_size=TEST_SIZE, random_state=SEED, stratify=stratif
)

adata_train = adata[idx_train].copy()
adata_val = adata[idx_val].copy()

print("train :", adata_train.shape)
print("val   :", adata_val.shape)

# proportions des types cellulaires dans chaque jeu
prop = pd.concat(
    [adata_train.obs["celltype"].value_counts(normalize=True).rename("train"),
     adata_val.obs["celltype"].value_counts(normalize=True).rename("val")],
    axis=1,
)
print(prop.round(3))

## 8. Filtre des gènes

On retire les gènes détectés dans moins de 3 cellules **du train**, puis on garde les mêmes gènes dans la validation.

In [ ]:
n_genes_avant = adata_train.n_vars
sc.pp.filter_genes(adata_train, min_cells=3)
adata_val = adata_val[:, adata_train.var_names].copy()
print(f"{n_genes_avant - adata_train.n_vars} gènes retirés, {adata_train.n_vars} restants")

## 9. Normalisation puis `log1p`

- **Normalisation** (`normalize_total`) : chaque cellule est ramenée à 10 000 comptages au total (CP10k), pour que la profondeur de séquençage ne fausse pas les comparaisons entre cellules.
- **`log1p`** : compresse l'échelle, car quelques gènes très exprimés (insuline...) écraseraient les autres. Le « +1 » évite log(0).
- On fixe `target_sum=1e4` : train et validation sont normalisés vers la même valeur (sinon chaque jeu utiliserait sa propre médiane).

In [ ]:
for a in (adata_train, adata_val):
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)

# contrôle : après expm1, chaque cellule doit sommer à environ 10 000
X = adata_train.X
X = X.expm1() if sp.issparse(X) else np.expm1(X)
print(np.asarray(X.sum(axis=1)).ravel()[:5])

## 10. Gènes hautement variables (HVG)

On garde les `N_HVG` gènes les plus variables, calculés **sur le train** et par technologie (`batch_key="batch"`) : un gène retenu doit être variable dans plusieurs technologies, pas seulement à cause de l'une d'elles. Les mêmes gènes sont ensuite gardés dans la validation.

In [ ]:
sc.pp.highly_variable_genes(adata_train, n_top_genes=N_HVG, batch_key="batch")
sc.pl.highly_variable_genes(adata_train)

genes = adata_train.var_names[adata_train.var["highly_variable"]]
print(len(genes), "gènes variables retenus")

adata_train = adata_train[:, genes].copy()
adata_val = adata_val[:, genes].copy()

## 11. Vérifications

In [ ]:
assert adata_train.n_vars == adata_val.n_vars == N_HVG
assert (adata_train.var_names == adata_val.var_names).all()
assert len(set(adata_train.obs_names) & set(adata_val.obs_names)) == 0, "cellules communes !"
print("train :", adata_train.shape, "| val :", adata_val.shape)

# des marqueurs connus du pancréas sont-ils retenus ?
marqueurs = ["INS", "GCG", "SST", "PPY", "PRSS1", "CPA1", "KRT19", "CFTR"]
for g in marqueurs:
    print(g, g in set(adata_train.var_names))

## 12. Sauvegarde

On sauvegarde **avant** toute PCA pour que les fichiers restent propres. Le notebook de l'auto-encodeur chargera `train.h5ad` et `val.h5ad`. Les layers `counts` (comptages bruts, mêmes cellules et mêmes gènes) sont conservées, utiles si on teste une perte NB/ZINB.

In [ ]:
adata_train.write_h5ad(f"{DATA_DIR}/train.h5ad")
adata_val.write_h5ad(f"{DATA_DIR}/val.h5ad")
np.save(f"{DATA_DIR}/idx_train.npy", idx_train)
np.save(f"{DATA_DIR}/idx_val.npy", idx_val)

for f in ["train.h5ad", "val.h5ad"]:
    print(f, round(os.path.getsize(f"{DATA_DIR}/{f}") / 1e6, 1), "Mo")

## 13. Visualisation : PCA du train

La PCA est calculée sur une **copie** du train (les fichiers sauvegardés ne la contiennent pas). Elle sert à regarder la structure des données et l'effet de batch. La PCA utilisée pour la comparaison avec l'auto-encodeur sera refaite avec scikit-learn, avec le même nombre de dimensions que l'AE.

In [ ]:
adata_vis = adata_train.copy()
sc.pp.pca(adata_vis, n_comps=50)

sc.pl.pca(adata_vis, color=["celltype", "batch"], size=5)
sc.pl.pca(
    adata_vis,
    color=["batch", "batch", "size_factors", "n_counts"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

In [ ]:
sc.pl.pca_variance_ratio(adata_vis, n_pcs=50, log=True)

## Résumé

- `data/train.h5ad` et `data/val.h5ad` : mêmes gènes (2000, dans le même ordre), cellules différentes.
- Gènes filtrés et sélectionnés sur le train uniquement, normalisation avec une cible fixe.
- Suite : notebook de l'auto-encodeur (entraînement sur le train, suivi sur la validation, comparaison avec la PCA).